In [ ]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")
os.environ["AZURE_STORAGE_CONNECTION_STRING"]=s.get_secret("AZURE_STORAGE_CONNECTION_STRING")


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("github")
repo_dir = "/kaggle/working/galaxy-classification"
url = "https://github.com/YubikNakarmi/galaxy-classification.git"

basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = {**os.environ,
       "GIT_TERMINAL_PROMPT": "0",
       "GIT_CONFIG_COUNT": "1",
       "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
       "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic}"}

def git(*args):
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed")

if not os.path.exists(repo_dir):
    git("clone", url, repo_dir)
else:
    git("-C", repo_dir, "checkout", "--", "configs/ml.yaml")   # drop local edits
    git("-C", repo_dir, "pull", url)

%cd /kaggle/working/galaxy-classification

In [ ]:
!ls

In [ ]:
!pip install uv

In [ ]:
!uv sync --frozen

In [ ]:
!uv pip install -e .

In [ ]:
import shutil

src = "/kaggle/input/datasets/yubiknakarmi/kaggle-config/ml_kaggle (4).yaml"
dst = "/kaggle/working/galaxy-classification/configs/ml.yaml"

shutil.copyfile(src, dst)

In [ ]:
!nvidia-smi

In [ ]:
%%capture out
!galax  pipeline train -e -y